# GlucoMeal AI — Complete ML Project Notebook

**Food-aware meal analysis, postprandial glucose response, D1NAMO training, PIMA external evaluation, and Indian food nutrition lookup**

This notebook mirrors the final GlucoMeal AI workflow:

1. Load the Indian food nutrition spreadsheet (`indian_food.xlsx`).
2. Load D1NAMO food/glucose data and align glucose around meal timestamps.
3. Calculate six-point response features at 0, 15, 40, 45, 60 and 120 minutes.
4. Train and compare Logistic Regression and Random Forest using participant-grouped cross-validation.
5. Evaluate a separate 2-hour-glucose model on PIMA as an external stress test.
6. Explore an example meal, calculate nutrition totals, and generate a prediction from six glucose readings.

> **Scientific limitation:** the Indian food spreadsheet contains nutrient information, not person-specific glucose outcomes or diabetes labels. Therefore its nutrients are used for meal nutrition analysis, not treated as learned diagnostic predictors. The six-point D1NAMO model predicts from the glucose response. PIMA has one 2-hour OGTT glucose measurement, not the six post-meal readings, so PIMA evaluation uses a separate glucose-only model and is not a direct validation of the six-point model.

## 1. Install dependencies

Run this cell once in the notebook environment. If you are using your existing virtual environment, the same packages can be installed with `pip install -r requirements.txt`.

In [ ]:
%pip install -q numpy pandas scipy scikit-learn matplotlib seaborn joblib openpyxl pyyaml

## 2. Configure file paths

Expected layout relative to this notebook:

```text
project/
├── GlucoMeal_AI_Final_Project.ipynb
└── data/
    ├── raw/
    │   ├── d1namo/
    │   │   ├── healthy_subset_pictures-glucose-food/
    │   │   └── diabetes_subset_pictures-glucose-food-insulin/
    │   └── pima/
    │       └── diabetes.csv
    └── food/
        └── indian_food.xlsx
```

Change the paths below if your folders are elsewhere.

In [ ]:
from pathlib import Path
import re
import warnings
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay
)

RANDOM_STATE = 42
ROOT = Path.cwd()
D1NAMO_ROOT = ROOT / "data" / "raw" / "d1namo"
PIMA_PATH = ROOT / "data" / "raw" / "pima" / "diabetes.csv"
FOOD_PATH = ROOT / "data" / "food" / "indian_food.xlsx"
OUTPUT_DIR = ROOT / "outputs"
MODEL_DIR = ROOT / "models"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
MODEL_DIR.mkdir(exist_ok=True, parents=True)

TIMEPOINTS = [0, 15, 40, 45, 60, 120]
print("Notebook root:", ROOT)
print("D1NAMO exists:", D1NAMO_ROOT.exists(), D1NAMO_ROOT)
print("PIMA exists:", PIMA_PATH.exists(), PIMA_PATH)
print("Indian food spreadsheet exists:", FOOD_PATH.exists(), FOOD_PATH)

## 3. Load and inspect the Indian food database

This section detects the spreadsheet columns and shows a preview. Nutrient names vary between food datasets, so inspect the printed column list and update the mapping in the next cell if necessary.

In [ ]:
if not FOOD_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find {FOOD_PATH}. Place indian_food.xlsx in data/food/ or update FOOD_PATH."
    )

food_raw = pd.read_excel(FOOD_PATH)
print("Rows:", len(food_raw))
print("Columns:", list(food_raw.columns))
display(food_raw.head())

## 4. Normalize food columns and calculate nutrition per serving

The code attempts to identify food names, serving-size, energy, carbohydrate, sugar, protein, fat and fibre columns. The input serving quantity is treated as grams when the database is expressed per 100 g. **Verify the source spreadsheet's units** before using the totals in a report.

In [ ]:
def norm_col(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).strip().lower()).strip("_")

food = food_raw.copy()
food.columns = [norm_col(c) for c in food.columns]

def find_col(columns, candidates):
    # exact match first
    for cand in candidates:
        if cand in columns:
            return cand
    # substring match second
    for cand in candidates:
        for col in columns:
            if cand in col:
                return col
    return None

food_name_col = find_col(food.columns, ["food_name", "name", "food", "dish", "item"])
if food_name_col is None:
    # Use first text-like column if names are not obvious.
    text_cols = food.select_dtypes(include=["object", "string"]).columns.tolist()
    food_name_col = text_cols[0] if text_cols else food.columns[0]

nutrient_map = {
    "energy_kcal": ["energy_kcal", "calories", "energy", "kcal"],
    "carbohydrate_g": ["carbohydrate_g", "carbohydrates", "carbohydrate", "carbs"],
    "sugar_g": ["free_sugar_g", "free_sugars", "sugar_g", "sugars", "sugar"],
    "protein_g": ["protein_g", "protein"],
    "fat_g": ["fat_g", "total_fat", "fat"],
    "fibre_g": ["dietary_fibre_g", "dietary_fiber", "fibre", "fiber"],
}
resolved_nutrients = {k: find_col(food.columns, v) for k, v in nutrient_map.items()}
print("Food name column:", food_name_col)
print("Nutrient column mapping:", resolved_nutrients)

food["food_name_display"] = food[food_name_col].astype(str)
for target, source in resolved_nutrients.items():
    if source is not None:
        food[target] = pd.to_numeric(food[source], errors="coerce")

display(food[["food_name_display"] + [c for c in nutrient_map if c in food.columns]].head())

## 5. Search foods and calculate a meal nutrition total

Edit the query to search the database. Then enter the food names and serving weights in grams in `meal_items`. Nutrition totals below assume nutrient values are per 100 g; if your spreadsheet uses another basis, adjust the multiplier.

In [ ]:
search_query = "rice"  # change to e.g. "chapati", "dal", "curd", "idli"
matches = food[food["food_name_display"].str.contains(search_query, case=False, na=False)]
display(matches[["food_name_display"] + [c for c in nutrient_map if c in food.columns]].head(20))

In [ ]:
# Replace the example names with exact/closest names from the food table.
# Serving amounts are grams. This is a nutrition-analysis example, not training labels.
meal_items = [
    {"food_query": "rice", "grams": 200},
    {"food_query": "dal", "grams": 150},
    {"food_query": "curd", "grams": 100},
]

def calculate_meal_nutrition(food_table, items):
    nutrient_cols = [c for c in nutrient_map if c in food_table.columns]
    rows = []
    for item in items:
        match = food_table[
            food_table["food_name_display"].str.contains(item["food_query"], case=False, na=False)
        ]
        if match.empty:
            print(f"WARNING: no food match for {item['food_query']!r}; skipped.")
            continue
        # Choose first match; inspect search results and make the query more specific if needed.
        r = match.iloc[0]
        grams = float(item["grams"])
        row = {"selected_food": r["food_name_display"], "grams": grams}
        for n in nutrient_cols:
            val = pd.to_numeric(r[n], errors="coerce")
            row[n] = float(val) * grams / 100.0 if pd.notna(val) else np.nan
        rows.append(row)
    details = pd.DataFrame(rows)
    totals = details[nutrient_cols].sum(min_count=1).to_dict() if not details.empty else {}
    return details, totals

meal_details, meal_totals = calculate_meal_nutrition(food, meal_items)
display(meal_details)
print("Estimated meal totals (assuming per-100-g nutrient values):")
display(pd.DataFrame([meal_totals]))

## 6. D1NAMO data loading and meal alignment

The loader searches recursively for CSV files with glucose in their filename and food/meal CSV files. It attempts to infer labels from parent folder names containing `healthy` or `diabetes`, clusters food timestamps into meal events, and samples glucose near each of the six target offsets.

**Please inspect the output carefully.** D1NAMO directory and timestamp formats can vary; if no samples are produced, inspect the CSV names/columns and adjust the parser rather than inventing readings.

In [ ]:
def find_datetime_column(df):
    low = {str(c).strip().lower(): c for c in df.columns}
    for name in ["datetime", "timestamp", "date_time", "time_stamp"]:
        if name in low:
            return low[name]
    return None

def parse_times(df):
    dt_col = find_datetime_column(df)
    if dt_col is not None:
        return pd.to_datetime(df[dt_col], errors="coerce", format="mixed")
    low = {str(c).strip().lower(): c for c in df.columns}
    date_col = next((low[x] for x in ["date", "day"] if x in low), None)
    time_col = next((low[x] for x in ["time", "clock"] if x in low), None)
    if date_col is not None and time_col is not None:
        return pd.to_datetime(
            df[date_col].astype(str) + " " + df[time_col].astype(str),
            errors="coerce", format="mixed"
        )
    if date_col is not None:
        return pd.to_datetime(df[date_col], errors="coerce", format="mixed")
    return pd.Series(pd.NaT, index=df.index)

def find_numeric_glucose_column(df):
    candidates = [c for c in df.columns if any(
        x in str(c).lower() for x in ["glucose", "blood_glucose", "bgl", "bg", "value", "concentration"]
    )]
    for c in candidates:
        if pd.api.types.is_numeric_dtype(df[c]):
            return c
    numeric = df.select_dtypes(include=np.number).columns.tolist()
    return numeric[0] if numeric else None

def read_glucose_csv(path):
    df = pd.read_csv(path)
    times = parse_times(df)
    val_col = find_numeric_glucose_column(df)
    if val_col is None:
        # Some CSVs contain numeric values stored as text.
        for c in df.columns:
            converted = pd.to_numeric(df[c], errors="coerce")
            if converted.notna().sum() > 0 and ("glucose" in str(c).lower() or c == df.columns[-1]):
                df[c] = converted
                val_col = c
                break
    if val_col is None:
        raise ValueError(f"No numeric glucose column found in {path}; columns={list(df.columns)}")
    vals = pd.to_numeric(df[val_col], errors="coerce")
    # D1NAMO glucose is normally mg/dL; convert mmol/L if values strongly suggest mmol/L.
    med = vals.dropna().median()
    if pd.notna(med) and med < 30:
        vals = vals * 18.0182
    out = pd.DataFrame({"timestamp": times, "glucose": vals}).dropna()
    return out.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)

def read_food_csv(path):
    df = pd.read_csv(path)
    times = parse_times(df)
    out = pd.DataFrame({"timestamp": times}).dropna()
    return out.sort_values("timestamp").reset_index(drop=True)

def label_from_path(path):
    p = str(path).lower()
    if "healthy" in p:
        return 0
    if "diabetes" in p or "diabetic" in p:
        return 1
    return None

def subject_from_path(path):
    # Prefer a 3-digit participant folder identifier if present.
    for part in reversed(Path(path).parts):
        m = re.search(r"(?<!\d)(\d{3})(?!\d)", part)
        if m:
            return m.group(1)
    return Path(path).parent.name

def cluster_meals(food_df, gap_minutes=30):
    if food_df.empty:
        return []
    ts = food_df["timestamp"].sort_values().reset_index(drop=True)
    group_ids = (ts.diff().dt.total_seconds().div(60).fillna(0) > gap_minutes).cumsum()
    return [g.median() for _, g in ts.groupby(group_ids)]

def nearest_glucose(gdf, target, tolerance_minutes=8):
    if gdf.empty:
        return np.nan
    differences = (gdf["timestamp"] - target).abs().dt.total_seconds() / 60
    idx = differences.idxmin()
    if differences.loc[idx] <= tolerance_minutes:
        return float(gdf.loc[idx, "glucose"])
    return np.nan

def response_features(values):
    x = np.asarray(values, dtype=float)
    if len(x) != 6 or not np.isfinite(x).all():
        return {}
    t = np.asarray(TIMEPOINTS, dtype=float)
    peak_i = int(np.argmax(x))
    auc_fn = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
    return {
        **{f"g_{m}": float(v) for m, v in zip(TIMEPOINTS, x)},
        "peak_glucose": float(x[peak_i]),
        "peak_time_min": float(t[peak_i]),
        "delta_120": float(x[-1] - x[0]),
        "peak_excursion": float(x[peak_i] - x[0]),
        "auc_0_120": float(auc_fn(x, t)),
        "slope_0_15": float((x[1] - x[0]) / 15),
        "slope_60_120": float((x[5] - x[4]) / 60),
        "recovery_60_120": float(x[4] - x[5]),
        "pct_peak_rise": float(100 * (x[peak_i] - x[0]) / max(x[0], 1e-6)),
    }

def load_d1namo_meals(root, tolerance_minutes=8):
    if not root.exists():
        raise FileNotFoundError(f"D1NAMO directory not found: {root}")
    glucose_files = [p for p in root.rglob("*.csv") if "glucose" in p.name.lower()]
    records = []
    print("Glucose CSV files found:", len(glucose_files))
    for gf in glucose_files:
        label = label_from_path(gf)
        if label is None:
            continue
        try:
            gdf = read_glucose_csv(gf)
        except Exception as exc:
            warnings.warn(f"Skipping glucose file {gf}: {exc}")
            continue
        parent = gf.parent
        food_files = [p for p in parent.rglob("*.csv")
                      if ("food" in p.name.lower() or "meal" in p.name.lower())]
        food_times = []
        for ff in food_files:
            try:
                food_times.extend(read_food_csv(ff)["timestamp"].tolist())
            except Exception as exc:
                warnings.warn(f"Skipping food file {ff}: {exc}")
        if not food_times:
            continue
        meal_times = cluster_meals(pd.DataFrame({"timestamp": pd.to_datetime(food_times)}))
        sid = subject_from_path(gf)
        for meal_time in meal_times:
            readings = [
                nearest_glucose(gdf, meal_time + pd.Timedelta(minutes=m), tolerance_minutes)
                for m in TIMEPOINTS
            ]
            if not np.isfinite(readings).all():
                continue
            feats = response_features(readings)
            if not feats:
                continue
            feats.update({"subject_id": sid, "label": int(label), "meal_time": meal_time})
            records.append(feats)
    result = pd.DataFrame(records)
    if result.empty:
        raise RuntimeError(
            "No complete six-point meal samples were extracted. Inspect the D1NAMO CSV structure, "
            "timestamp columns, and food annotations. Do not fabricate missing readings."
        )
    return result

# This can take a few minutes depending on the extracted dataset size.
d1namo_samples = load_d1namo_meals(D1NAMO_ROOT)
d1namo_samples.to_csv(OUTPUT_DIR / "d1namo_meal_samples_notebook.csv", index=False)
print("Meal samples:", len(d1namo_samples))
print("Participants:", d1namo_samples["subject_id"].nunique())
display(d1namo_samples.groupby("label").agg(
    rows=("subject_id", "size"), participants=("subject_id", "nunique")
))

## 7. Inspect the extracted glucose-response data

In [ ]:
display(d1namo_samples.head())
availability = d1namo_samples[[f"g_{m}" for m in TIMEPOINTS]].notna().mean().mul(100)
display(availability.rename("available_percent").to_frame())

plt.figure(figsize=(9, 5))
for label, group in d1namo_samples.groupby("label"):
    cols = [f"g_{m}" for m in TIMEPOINTS]
    mean_curve = group[cols].mean()
    plt.plot(TIMEPOINTS, mean_curve.values, marker="o", label=("Healthy" if label == 0 else "Diabetes cohort"))
plt.xlabel("Minutes after meal")
plt.ylabel("Glucose (mg/dL)")
plt.title("D1NAMO mean glucose response by dataset label")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

## 8. Train six-point response classifiers with participant-level validation

Meals from the same person are correlated, so we group by `subject_id` in cross-validation. This is important to reduce participant leakage. Given the small D1NAMO cohort, the metrics should be interpreted as preliminary rather than clinically reliable.

In [ ]:
MODEL_FEATURES = [
    "g_0", "g_15", "g_40", "g_45", "g_60", "g_120",
    "peak_glucose", "peak_time_min", "delta_120", "peak_excursion",
    "auc_0_120", "slope_0_15", "slope_60_120", "recovery_60_120",
    "pct_peak_rise",
]
X = d1namo_samples[MODEL_FEATURES].copy()
y = d1namo_samples["label"].astype(int)
groups = d1namo_samples["subject_id"].astype(str)

class_subject_counts = d1namo_samples.drop_duplicates("subject_id").groupby("label")["subject_id"].nunique()
n_splits = min(5, int(class_subject_counts.min()))
if n_splits < 2:
    raise ValueError("Not enough participants in both classes for grouped cross-validation.")
print("Grouped CV folds:", n_splits)
print("Participants per label:", class_subject_counts.to_dict())

def build_models():
    return {
        "Logistic Regression": Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE)),
        ]),
        "Random Forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("model", RandomForestClassifier(
                n_estimators=500, max_depth=5, min_samples_leaf=3,
                class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
            )),
        ]),
    }

def grouped_cv_metrics(estimator, X, y, groups, n_splits):
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    yt, yp, pp = [], [], []
    for tr, va in cv.split(X, y, groups):
        model = clone(estimator)
        model.fit(X.iloc[tr], y.iloc[tr])
        prob = model.predict_proba(X.iloc[va])[:, 1]
        pred = (prob >= 0.5).astype(int)
        yt.extend(y.iloc[va].tolist())
        yp.extend(pred.tolist())
        pp.extend(prob.tolist())
    return {
        "accuracy": accuracy_score(yt, yp),
        "balanced_accuracy": balanced_accuracy_score(yt, yp),
        "precision": precision_score(yt, yp, zero_division=0),
        "recall": recall_score(yt, yp, zero_division=0),
        "f1": f1_score(yt, yp, zero_division=0),
        "roc_auc": roc_auc_score(yt, pp),
    }

candidate_models = build_models()
cv_results = {}
for name, estimator in candidate_models.items():
    cv_results[name] = grouped_cv_metrics(estimator, X, y, groups, n_splits)

cv_table = pd.DataFrame(cv_results).T.sort_values(["roc_auc", "balanced_accuracy"], ascending=False)
display(cv_table.round(4))
best_name = cv_table.index[0]
best_model = build_models()[best_name]
best_model.fit(X, y)
joblib.dump(best_model, MODEL_DIR / "glucose_response_model.joblib")
with open(OUTPUT_DIR / "d1namo_cv_metrics.json", "w") as f:
    json.dump(cv_table.to_dict(orient="index"), f, indent=2)
print("Selected model:", best_name)
print("Saved:", MODEL_DIR / "glucose_response_model.joblib")

## 9. Evaluate a separate 2-hour glucose model on PIMA

PIMA's `Glucose` column is a 2-hour oral glucose tolerance test measurement. It is not the same protocol as D1NAMO's post-meal readings. To avoid pretending PIMA has six time points, this section trains a separate D1NAMO model using only `g_120` and applies it to PIMA's `Glucose`. Treat this as a cross-dataset stress test, not a direct validation of the six-point model.

In [ ]:
if not PIMA_PATH.exists():
    print("PIMA file not found; skipping external evaluation:", PIMA_PATH)
else:
    pima = pd.read_csv(PIMA_PATH)
    rename = {}
    for c in pima.columns:
        key = str(c).strip().lower().replace("_", "").replace(" ", "")
        if key == "glucose":
            rename[c] = "Glucose"
        elif key == "outcome":
            rename[c] = "Outcome"
    pima = pima.rename(columns=rename)
    if not {"Glucose", "Outcome"}.issubset(pima.columns):
        raise ValueError(f"PIMA CSV must have Glucose and Outcome columns. Found: {list(pima.columns)}")

    # Build a separate compatible model using only D1NAMO 2-hour glucose.
    X2 = d1namo_samples[["g_120"]]
    y2 = d1namo_samples["label"].astype(int)
    groups2 = d1namo_samples["subject_id"].astype(str)
    pima_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE)),
    ])
    pima_model.fit(X2, y2)
    joblib.dump(pima_model, MODEL_DIR / "d1namo_2hour_glucose_model.joblib")

    pima_X = pima[["Glucose"]].rename(columns={"Glucose": "g_120"})
    pima_y = pima["Outcome"].astype(int)
    pima_prob = pima_model.predict_proba(pima_X)[:, 1]
    pima_pred = (pima_prob >= 0.5).astype(int)

    pima_metrics = {
        "accuracy": accuracy_score(pima_y, pima_pred),
        "balanced_accuracy": balanced_accuracy_score(pima_y, pima_pred),
        "precision": precision_score(pima_y, pima_pred, zero_division=0),
        "recall": recall_score(pima_y, pima_pred, zero_division=0),
        "f1": f1_score(pima_y, pima_pred, zero_division=0),
        "roc_auc": roc_auc_score(pima_y, pima_prob),
    }
    display(pd.DataFrame([pima_metrics]).round(4))
    print(classification_report(pima_y, pima_pred, zero_division=0))
    print("Confusion matrix:\\n", confusion_matrix(pima_y, pima_pred))
    with open(OUTPUT_DIR / "pima_external_metrics.json", "w") as f:
        json.dump(pima_metrics, f, indent=2)

## 10. Predict from six user-entered glucose values

The following cell demonstrates the same six-point feature construction used by the final model. It prints the estimated model probability and the response summary. This is a model output, **not a diagnosis**.

In [ ]:
def summarize_six_readings(values):
    x = np.asarray(values, dtype=float)
    if len(x) != 6 or not np.isfinite(x).all():
        raise ValueError("Enter six finite glucose readings in order 0, 15, 40, 45, 60, 120 minutes.")
    t = np.asarray(TIMEPOINTS, dtype=float)
    peak_idx = int(np.argmax(x))
    auc_fn = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
    return {
        **{f"g_{m}": float(v) for m, v in zip(TIMEPOINTS, x)},
        "peak_glucose": float(x[peak_idx]),
        "peak_time_min": float(t[peak_idx]),
        "delta_120": float(x[-1] - x[0]),
        "peak_excursion": float(x[peak_idx] - x[0]),
        "auc_0_120": float(auc_fn(x, t)),
        "slope_0_15": float((x[1] - x[0]) / 15),
        "slope_60_120": float((x[5] - x[4]) / 60),
        "recovery_60_120": float(x[4] - x[5]),
        "pct_peak_rise": float(100 * (x[peak_idx] - x[0]) / max(x[0], 1e-6)),
    }

# Example values: replace these with the user's six measurements.
sample_readings = [125, 150, 180, 190, 200, 175]
sample_features = summarize_six_readings(sample_readings)
sample_X = pd.DataFrame([{c: sample_features[c] for c in MODEL_FEATURES}])
sample_probability = float(best_model.predict_proba(sample_X)[0, 1])

print(f"Model-estimated positive-class probability: {sample_probability:.3f}")
print("This is an academic model estimate, not a clinical diagnosis.")
display(pd.DataFrame([sample_features]))

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(TIMEPOINTS, sample_readings, marker="o")
plt.xlabel("Minutes after meal")
plt.ylabel("Glucose (mg/dL)")
plt.title("Sample post-meal glucose response")
plt.grid(alpha=0.25)
plt.show()

## 11. Compare the response with general reference values

This comparison is descriptive only. A common post-meal glucose management target for many nonpregnant adults with diabetes is below 180 mg/dL at 1–2 hours after the beginning of a meal. This is **not a diagnostic cutoff**, and the user's clinical context may differ. The six-point readings also start immediately after the meal here, whereas the clinical reference is timed from the beginning of the meal.

In [ ]:
def reference_summary(values):
    x = np.asarray(values, dtype=float)
    return {
        "1-hour reading (mg/dL)": x[4],
        "2-hour reading (mg/dL)": x[5],
        "1-hour below 180 reference": bool(x[4] < 180),
        "2-hour below 180 reference": bool(x[5] < 180),
        "note": "General management reference only; not a diagnostic test."
    }

display(pd.DataFrame([reference_summary(sample_readings)]))

## 12. Save final artifacts

The notebook writes the extracted D1NAMO meal table, model files, and evaluation metrics into `outputs/` and `models/`.

In [ ]:
print("Models:")
for p in sorted(MODEL_DIR.glob("*")):
    print(" -", p.name, f"({p.stat().st_size:,} bytes)")
print("\\nOutputs:")
for p in sorted(OUTPUT_DIR.glob("*")):
    if p.is_file():
        print(" -", p.name, f"({p.stat().st_size:,} bytes)")

## 13. Limitations and responsible interpretation

- D1NAMO is a small cohort and its diabetes participants have Type 1 diabetes; this does not represent every type of diabetes.
- Meal-level labels inherit participant diagnosis, so the model learns cohort differences and is not proof that a single meal can diagnose diabetes.
- Healthy and diabetic D1NAMO recordings may have different glucose measurement density; only complete six-point records are used here.
- PIMA uses a 2-hour OGTT glucose measurement, not a six-point post-meal curve.
- The Indian food table is a nutrient database, not a dataset of meal-specific glucose responses.
- Food nutrient totals depend on correct serving units and food matching.
- Do not use this notebook or model to make medical decisions. Clinical diagnosis requires validated tests and qualified healthcare professionals.